# Exercise 3 – Clustering Unlabelled MMS Ion Spectra

**Companion notebook:** `03_pca_clustering_plasma_regions.ipynb`

**Companion dataset:** `data/ex1_cleaned_unlabelled.nc` (produced in Exercise 1)

Work through the cells in order. Each exercise has a **Problem** statement in markdown followed by a code cell with `# YOUR CODE HERE`.


In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from spacephyml.datasets.mms import SpectrumDataset

sns.set_theme()
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

figure_path = Path('./figures')
figure_path.mkdir(exist_ok=True)

RANDOM_STATE = 42
NC_FILE = 'data/ex1_cleaned_unlabelled.nc'

---

## Background

In the companion notebook, clustering was treated as a verification exercise: you already knew there were four plasma regions, you set `n_clusters = 4`, ran the algorithms, and measured how well the recovered clusters matched the known labels with ARI and NMI.

In this exercise you face the harder, more realistic problem: you have the **same unlabelled dataset from Exercise 2**, and you do not know in advance how many clusters the data contains. You must decide:

1. How many clusters to ask for.
2. Which algorithm to use.
3. Whether the result is physically meaningful.

Without ground-truth labels, you cannot use ARI or NMI. Instead you will use **internal validation metrics** — scores computed entirely from the data and the cluster assignments — and **physical reasoning** from the orbital context and PC-space structure established in Exercise 2.

The three internal metrics used here are:

| Metric | What it measures | Better when |
|--------|-----------------|-------------|
| **Silhouette score** | How similar each point is to its own cluster vs. the nearest other cluster. Ranges from −1 to 1. | Higher (→ 1) |
| **Davies–Bouldin index** | Average ratio of within-cluster scatter to between-cluster separation. | Lower (→ 0) |
| **Calinski–Harabász index** | Ratio of between-cluster to within-cluster dispersion. | Higher |

No single metric is definitive. The goal is to find a *k* and algorithm where multiple metrics agree and the resulting clusters make physical sense.

---

### Exercise 3.1 – Prepare the PCA-reduced feature matrix

Load the cleaned unlabelled dataset (5-minute windows, `N = 66`, all available samples) and prepare the feature matrix that the clustering algorithms will use:

1. Load with `flatten=True`.
2. Apply `log10p1` preprocessing.
3. Fit PCA on the **full** dataset (there is no train/test split here — why not?) and project onto the top 8 components.
4. Print the shape of the resulting array and the variance retained.

> **Think about it:** In the companion notebook, PCA was fitted only on the training set to avoid data leakage. Here, there is no downstream supervised model, so there are no held-out test labels to leak into. Fitting PCA on all available unlabelled data gives the most stable eigenvectors.

In [ ]:
N      = 66    # ≈ 5 minutes at 4.5 s cadence
N_PCS  = 8

def preprocess(X: np.ndarray) -> np.ndarray:
    """Global log₁₀(1 + x) scaling — preserves spectral shape."""
    return np.log10(1. + X)

# YOUR CODE HERE
# 1. Create a SpectrumDataset from NC_FILE, N=66, flatten=True.
# 2. Apply preprocess().
# 3. Fit PCA(n_components=N_PCS, random_state=RANDOM_STATE) on the full matrix.
# 4. Project: X_pca = pca.transform(X_scaled)
# 5. Print X_pca.shape and the cumulative variance retained.


---

### Exercise 3.2 – Sweep over k: inertia and internal metrics for K-Means

Fit K-Means for `k = 2, 3, 4, 5, 6, 7, 8`. For each k, record:

- **Inertia** (within-cluster sum of squares — available as `kmeans.inertia_`): https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html
- **Silhouette score**: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html
- **Davies–Bouldin index**: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.davies_bouldin_score.html
- **Calinski–Harabász index**: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.calinski_harabasz_score.html

Plot all four metrics in a 2×2 figure (one panel per metric, x-axis = k). Add a vertical dashed line at the k you think is best based on a visual elbow or peak, and annotate it.

Use `n_init=20` and `max_iter=500` for stable results.

In [ ]:
K_RANGE = range(2, 9)

# YOUR CODE HERE
# For each k in K_RANGE:
#   1. Fit KMeans(n_clusters=k, n_init=20, max_iter=500, random_state=RANDOM_STATE)
#      on X_pca.
#   2. Record inertia_, silhouette_score, davies_bouldin_score,
#      calinski_harabasz_score.
#      (Tip: silhouette_score is slow for large N; use sample_size=2000 if needed)
#
# Store results in a dict or DataFrame, then make a 2×2 figure.
# Save to figure_path / '03a_kmeans_sweep.png'.


---

### Exercise 3.3 – Sweep over k: BIC and AIC for GMM

Fit a `GaussianMixture` with `covariance_type='full'` for the same range `k = 2, …, 8`. Record the **BIC** and **AIC** for each k.

BIC and AIC penalise model complexity (more components = more parameters), so the minimum of each curve is preferred. Plot BIC and AIC on the same axes (two lines, one legend). Add a vertical dashed line at the minimum BIC. https://scikit-learn.org/stable/auto_examples/mixture/plot_gmm_selection.html

> **Why BIC rather than silhouette for GMM?** GMM is a probabilistic model — it has a likelihood that can be compared across different numbers of components on the same data. BIC is the natural model-selection criterion. For K-Means (no likelihood), inertia and silhouette fill the same role.

In [ ]:
# YOUR CODE HERE
# For each k in K_RANGE:
#   1. Fit GaussianMixture(n_components=k, covariance_type='full',
#                          n_init=5, max_iter=300, random_state=RANDOM_STATE)
#      on X_pca.
#   2. Record gmm.bic(X_pca) and gmm.aic(X_pca).
#
# Plot BIC and AIC vs k on the same axes.
# Add a vertical dashed line at the k that minimises BIC.
# Save to figure_path / '03a_gmm_bic_aic.png'.


---

### Exercise 3.4 – Choose your best k and fit all three algorithms

Based on Exercises 3.2 and 3.3, choose a single value of `k`. Write one or two sentences below explaining your choice (which metrics pointed to it, and whether it aligns with your physical expectation from the known number of dayside plasma regions).

Then fit all three algorithms at that k:

- `KMeans(n_clusters=k, n_init=20, max_iter=500)`
- `GaussianMixture(n_components=k, covariance_type='full', n_init=5)`
- `AgglomerativeClustering(n_clusters=k, linkage='ward')` — fit on the full `X_pca` since it has no `predict()`

Collect the cluster assignments for all three into a dictionary `labels_dict`.

*(Edit this cell — explain your choice of k)*

Chosen k: 

Justification: 

In [ ]:
K_BEST = 4   # replace with your chosen value

# YOUR CODE HERE
# Fit KMeans, GaussianMixture, and AgglomerativeClustering at K_BEST.
# Store assignments in:
#   labels_dict = {'K-Means': ..., 'GMM': ..., 'Agglomerative': ...}
# Each value should be a 1-D integer array of length len(X_pca).


---

### Exercise 3.5 – Compare internal scores across all three algorithms

Compute the silhouette score, Davies–Bouldin index, and Calinski–Harabász index for each algorithm's cluster assignments at your chosen k. Display the results as a formatted table (a `pd.DataFrame` printed with `.to_string()` is fine).

Which algorithm produces the most internally coherent clusters according to each metric? Do all three metrics agree?

In [ ]:
# YOUR CODE HERE
# For each algorithm in labels_dict:
#   Compute silhouette_score, davies_bouldin_score, calinski_harabasz_score
#   on X_pca with the corresponding labels.
# Collect into a DataFrame with columns ['Silhouette ↑', 'Davies-Bouldin ↓', 'Calinski-Harabász ↑']
# and print it.


---

### Exercise 3.6 – Visualise cluster assignments in PC space

Make a 1×3 figure (one panel per algorithm) showing the cluster assignments in the PC1–PC2 projection. Each cluster should have a distinct colour; use a qualitative colormap (e.g. `tab10`). Add a legend with cluster IDs.

Below the figure, write a one-sentence observation for each algorithm: do the clusters correspond to visually distinct regions in PC space, or do some algorithms split a single cloud arbitrarily?

In [ ]:
# YOUR CODE HERE
# Make a 1×3 figure; in each panel scatter X_pca[:, 0] vs X_pca[:, 1]
# coloured by the cluster assignment from that algorithm.
# Label axes with variance-explained percentages as in Exercise 2.5.
# Save to figure_path / '03a_cluster_pc_space.png'.


---

### Exercise 3.7 – Interpret clusters using the log mean flux proxy

In Exercise 2.6 you used the log mean flux as a proxy for the plasma region. Now use it to give each cluster a tentative physical label.

For each algorithm and each cluster, compute the **mean log mean flux** of all windows in that cluster. Sort the clusters by this value.

Print a table with columns: `Algorithm | Cluster | Mean log flux | Tentative label`. Use this to annotate the PC-space plot from Exercise 3.6 (replace numeric cluster IDs in the legend with your tentative labels).

> **Note:** This is an informal heuristic, not a validated classification. The goal is to connect the cluster structure to physical context, not to replace the labelling work from the companion notebook.

In [ ]:
# Recompute log mean flux for each window (already in X_scaled from 3.1)
log_mean_flux = X_scaled.mean(axis=1)   # shape: (n_windows,)

# Tentative label assignment by flux rank:
# Lowest flux  → Magnetosphere
# Middle flux  → Magnetosheath  (if k >= 3)
# Higher flux  → Ion Foreshock  (if k >= 4)
# Highest flux → Solar Wind
TENTATIVE = ['Magnetosphere', 'Magnetosheath', 'Ion Foreshock', 'Solar Wind']

# YOUR CODE HERE
# For each algorithm in labels_dict:
#   1. For each cluster id (0 … K_BEST-1):
#      a. Select the windows in that cluster.
#      b. Compute mean log mean flux for that cluster.
#   2. Rank clusters by mean log flux and assign tentative labels from TENTATIVE.
#   3. Print the table described above.
#
# Then re-make the 1×3 PC-space scatter using tentative labels as the legend.
# Save to figure_path / '03a_cluster_pc_labelled.png'.


---

### Exercise 3.8 – Sensitivity to k: what breaks at k = 2 and k = 8?

Fit **GMM** at `k = 2` and `k = 8` and plot the cluster assignments in PC1–PC2 space side by side (alongside your chosen `k` from Exercise 3.4 for comparison — three panels total).

Describe in the markdown cell below:

1. At `k = 2`: which physical populations are merged together? Does the split at 2 clusters still reflect a physically meaningful boundary?
2. At `k = 8`: which of the known regions appear to be split into sub-clusters? Is this splitting physically interpretable (e.g. a sub-structure within the magnetosheath) or does it look like the algorithm is over-partitioning a single continuous distribution?

In [ ]:
# YOUR CODE HERE
# Fit GMM at k=2 and k=6 on X_pca.
# Make a 1×3 figure: k=2 | k=K_BEST | k=6
# Each panel: PC1 vs PC2 scatter coloured by cluster assignment.
# Save to figure_path / '03a_gmm_k_sensitivity.png'.


*(Double-click to edit — write your observations here)*

**k = 2:**

**k = 8:**

---

### Exercise 3.9 – Reflection: internal metrics vs physical interpretability

Answer the following questions in the markdown cell below:

1. Did the internal metrics (silhouette, DB, CH, BIC) all agree on the same best k? If not, how did you resolve the disagreement?
2. The companion notebook used ARI and NMI to evaluate clustering quality. You could not use those here. What did you lose by not having labels, and what did you gain (if anything)?
3. A colleague suggests simply picking k = 4 because "we know there are four plasma regions." What is the danger of this approach when working with unlabelled data from a new mission or orbit phase that you have not yet characterised?

*(Double-click to edit — write your answers here)*

1. 

2. 

3. 

---

## Summary

In this exercise you applied unsupervised clustering to the unlabelled March 2018 MMS dataset:

- Without ground-truth labels, **internal validation metrics** (silhouette, Davies–Bouldin, Calinski–Harabász, BIC) replace ARI and NMI as the primary tools for selecting k and comparing algorithms.
- No single metric is definitive. Physical reasoning — how many plasma regions does MMS encounter on a dayside orbit? — provides an important prior that should inform, but not override, the data-driven choice.
- The **log mean flux proxy** offers a lightweight way to assign tentative physical labels to clusters, connecting unsupervised output to interpretable science even in the absence of ground-truth labels.
- **k sensitivity analysis** (Exercises 3.2 and 3.8) reveals that too few clusters merge physically distinct populations, while too many split a single distribution arbitrarily. The "right" k lies in the range where metrics stabilise and the PC-space structure is cleanly partitioned.
- **GMM** tends to outperform K-Means on this data because the plasma regions form elongated, overlapping distributions in PC space — consistent with the findings of Toy-Edens et al. (2024).

**Key principle:** Internal metrics tell you how well the data is partitioned; they cannot tell you whether the partition is physically meaningful. Always pair quantitative scores with visual inspection and domain knowledge.

### Further reading
- Toy-Edens et al. (2024) — GMM clustering of 8 years of MMS dayside regions: https://doi.org/10.1029/2024JA032431
- Rousseeuw (1987) — Silhouettes: a graphical aid to the interpretation and validation of cluster analysis: https://doi.org/10.1016/0377-0427(87)90125-7
- Davies & Bouldin (1979) — A cluster separation measure: https://doi.org/10.1109/TPAMI.1979.4766909
